# PHẦN B — PHÂN LOẠI CHỮ SỐ MNIST BẰNG CNN

**Assignment 04**

Các bước:
1. Load MNIST 
2. Code NumPy CNN từ đầu
3. Code Keras CNN
4. Code PyTorch CNN
5. Đánh giá và lưu lịch sử

## 1. Import Thư Viện

In [1]:
import os, struct, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

# Set random seed
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Load Dữ liệu MNIST

In [2]:
def read_images(path):
    with open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(n, rows, cols)

def read_labels(path):
    with open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

DATA_DIR = '../MNIST'
train_images = read_images(os.path.join(DATA_DIR, 'train-images-idx3-ubyte'))
train_labels = read_labels(os.path.join(DATA_DIR, 'train-labels-idx1-ubyte'))
test_images = read_images(os.path.join(DATA_DIR, 't10k-images-idx3-ubyte'))
test_labels = read_labels(os.path.join(DATA_DIR, 't10k-labels-idx1-ubyte'))

# Chia tập validation (80-20)
order = np.random.permutation(len(train_images))
cut = int(0.8 * len(order))

# Thêm kênh màu (dim 3), rescale về [0, 1]
x_train_np = train_images[order[:cut]][..., None].astype('float32') / 255.0
y_train_np = train_labels[order[:cut]]
x_val_np = train_images[order[cut:]][..., None].astype('float32') / 255.0
y_val_np = train_labels[order[cut:]]
x_test_np = test_images[..., None].astype('float32') / 255.0
y_test_np = test_labels

print(f"Train: {x_train_np.shape}, {y_train_np.shape}")
print(f"Val:   {x_val_np.shape}, {y_val_np.shape}")
print(f"Test:  {x_test_np.shape}, {y_test_np.shape}")


Train: (48000, 28, 28, 1), (48000,)
Val:   (12000, 28, 28, 1), (12000,)
Test:  (10000, 28, 28, 1), (10000,)


### 2.1. Hiển thị thử dữ liệu

In [3]:
fig, axes = plt.subplots(1, 5, figsize=(12,3))
for i, ax in enumerate(axes):
    ax.imshow(x_train_np[i, ..., 0], cmap='gray')
    ax.set_title(f'Label: {y_train_np[i]}')
    ax.axis('off')
plt.show()


## 3. NumPy CNN from Scratch

Đây là các module cơ bản giống phần A, được điều chỉnh cho 1 kênh màu và 10 lớp output.

In [4]:
class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, random_state=42):
        self.kernel_size = kernel_size
        self.padding = padding
        rng = np.random.default_rng(random_state)
        fan_in = kernel_size * kernel_size * in_channels
        self.weight = rng.normal(0.0, np.sqrt(2.0/fan_in), size=(kernel_size, kernel_size, in_channels, out_channels)).astype(np.float32)
        self.bias = np.zeros(out_channels, dtype=np.float32)
        self.grad_weight = None
        self.grad_bias = None

    def _extract_patches(self, x):
        p = self.padding
        x_padded = np.pad(x, ((0,0), (p,p), (p,p), (0,0)), mode='constant')
        windows = np.lib.stride_tricks.sliding_window_view(x_padded, (self.kernel_size, self.kernel_size), axis=(1,2))
        return x_padded, windows.transpose(0,1,2,4,5,3)

    def forward(self, x):
        self.x_padded, self.patches = self._extract_patches(x)
        return np.einsum('nhwklc,klco->nhwo', self.patches, self.weight, optimize=True) + self.bias

    def backward(self, grad_output):
        self.grad_weight = np.einsum('nhwklc,nhwo->klco', self.patches, grad_output, optimize=True)
        self.grad_bias = grad_output.sum(axis=(0,1,2))
        grad_patches = np.einsum('nhwo,klco->nhwklc', grad_output, self.weight, optimize=True)
        grad_padded = np.zeros_like(self.x_padded)
        out_h, out_w = grad_output.shape[1:3]
        for row in range(self.kernel_size):
            for col in range(self.kernel_size):
                grad_padded[:, row:row+out_h, col:col+out_w, :] += grad_patches[:,:,:,row,col,:]
        p = self.padding
        return grad_padded if p == 0 else grad_padded[:, p:-p, p:-p, :]


In [5]:
class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return np.maximum(x, 0)
    def backward(self, grad_output):
        return grad_output * self.mask

class MaxPool2D:
    def __init__(self, pool_size=2):
        self.pool_size = pool_size
    def forward(self, x):
        n, h, w, c = x.shape
        p = self.pool_size
        self.input_shape = x.shape
        self.blocks = x.reshape(n, h//p, p, w//p, p, c)
        output = self.blocks.max(axis=(2,4))
        self.max_mask = self.blocks == output[:,:,None,:,None,:]
        self.tie_count = self.max_mask.sum(axis=(2,4), keepdims=True)
        return output
    def backward(self, grad_output):
        distributed = self.max_mask * grad_output[:,:,None,:,None,:] / self.tie_count
        return distributed.reshape(self.input_shape)

class Flatten:
    def forward(self, x):
        self.input_shape = x.shape
        return x.reshape(x.shape[0], -1)
    def backward(self, grad_output):
        return grad_output.reshape(self.input_shape)

class Linear:
    def __init__(self, in_features, out_features, random_state=43):
        rng = np.random.default_rng(random_state)
        self.weight = rng.normal(0.0, np.sqrt(2.0/in_features), size=(in_features, out_features)).astype(np.float32)
        self.bias = np.zeros(out_features, dtype=np.float32)
        self.grad_weight = None
        self.grad_bias = None
    def forward(self, x):
        self.input = x
        return x @ self.weight + self.bias
    def backward(self, grad_output):
        self.grad_weight = self.input.T @ grad_output
        self.grad_bias = grad_output.sum(axis=0)
        return grad_output @ self.weight.T

class SoftmaxCrossEntropy:
    def forward(self, logits, labels):
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp_values = np.exp(shifted)
        self.probabilities = exp_values / exp_values.sum(axis=1, keepdims=True)
        self.labels = labels.astype(np.int64)
        correct = self.probabilities[np.arange(len(labels)), self.labels]
        return float(-np.log(np.clip(correct, 1e-9, 1.0)).mean())
    def backward(self):
        gradient = self.probabilities.copy()
        gradient[np.arange(len(self.labels)), self.labels] -= 1.0
        return gradient / len(self.labels)

class NumpyAdam:
    def __init__(self, learning_rate=1e-3, weight_decay=1e-4, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.beta1, self.beta2, self.epsilon = beta1, beta2, epsilon
        self.first_moment, self.second_moment = {}, {}
        self.step_count = 0
    def step(self, named_parameters):
        self.step_count += 1
        for name, parameter, gradient, apply_decay in named_parameters:
            gradient = np.clip(gradient, -5.0, 5.0)
            if apply_decay: gradient += self.weight_decay * parameter
            if name not in self.first_moment:
                self.first_moment[name] = np.zeros_like(parameter)
                self.second_moment[name] = np.zeros_like(parameter)
            first = self.first_moment[name]
            second = self.second_moment[name]
            first *= self.beta1
            first += (1.0 - self.beta1) * gradient
            second *= self.beta2
            second += (1.0 - self.beta2) * gradient * gradient
            first_hat = first / (1.0 - self.beta1 ** self.step_count)
            second_hat = second / (1.0 - self.beta2 ** self.step_count)
            parameter -= self.learning_rate * first_hat / (np.sqrt(second_hat) + self.epsilon)


### 3.1. Xây dựng Numpy CNN Architecture

In [6]:
class NumpyCNNMnist:
    def __init__(self, input_shape=(28, 28, 1), conv_filters=(8, 16), hidden_units=64, num_classes=10):
        h, w, c = input_shape
        f1, f2 = conv_filters
        self.conv1 = Conv2D(c, f1)
        self.relu1 = ReLU()
        self.pool1 = MaxPool2D(2)
        
        self.conv2 = Conv2D(f1, f2)
        self.relu2 = ReLU()
        self.pool2 = MaxPool2D(2)
        
        self.flatten = Flatten()
        flat_size = (h // 4) * (w // 4) * f2
        self.hidden = Linear(flat_size, hidden_units)
        self.relu3 = ReLU()
        self.output = Linear(hidden_units, num_classes)

    def forward(self, x):
        x = self.pool1.forward(self.relu1.forward(self.conv1.forward(x)))
        x = self.pool2.forward(self.relu2.forward(self.conv2.forward(x)))
        x = self.flatten.forward(x)
        x = self.relu3.forward(self.hidden.forward(x))
        return self.output.forward(x)

    def backward(self, grad_logits):
        grad = self.output.backward(grad_logits)
        grad = self.relu3.backward(grad)
        grad = self.hidden.backward(grad)
        grad = self.flatten.backward(grad)
        grad = self.pool2.backward(grad)
        grad = self.relu2.backward(grad)
        grad = self.conv2.backward(grad)
        grad = self.pool1.backward(grad)
        grad = self.relu1.backward(grad)
        return self.conv1.backward(grad)

    def named_parameters(self):
        return [
            ("conv1.weight", self.conv1.weight, self.conv1.grad_weight, True),
            ("conv1.bias", self.conv1.bias, self.conv1.grad_bias, False),
            ("conv2.weight", self.conv2.weight, self.conv2.grad_weight, True),
            ("conv2.bias", self.conv2.bias, self.conv2.grad_bias, False),
            ("hidden.weight", self.hidden.weight, self.hidden.grad_weight, True),
            ("hidden.bias", self.hidden.bias, self.hidden.grad_bias, False),
            ("output.weight", self.output.weight, self.output.grad_weight, True),
            ("output.bias", self.output.bias, self.output.grad_bias, False),
        ]
    
    def param_count(self):
        return sum(p.size for _, p, _, _ in self.named_parameters() if len(p.shape) > 1)


### 3.2. Huấn luyện NumPy CNN

In [7]:
def train_numpy(model, X, y, X_val, y_val, epochs=3, batch_size=128, lr=1e-3):
    loss_fn = SoftmaxCrossEntropy()
    optimizer = NumpyAdam(learning_rate=lr)
    history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
    
    rng = np.random.default_rng(42)
    for epoch in range(epochs):
        ids = rng.permutation(len(y))
        total_loss = correct = 0
        
        for start in range(0, len(ids), batch_size):
            batch = ids[start:start+batch_size]
            logits = model.forward(X[batch])
            loss = loss_fn.forward(logits, y[batch])
            
            model.backward(loss_fn.backward())
            optimizer.step(model.named_parameters())
            
            total_loss += loss * len(batch)
            correct += np.sum(logits.argmax(1) == y[batch])
            
        train_loss = total_loss / len(y)
        train_accuracy = correct / len(y)
        
        # Đánh giá validation
        val_logits = []
        for start in range(0, len(y_val), 256):
            val_logits.append(model.forward(X_val[start:start+256]))
        val_logits = np.concatenate(val_logits, axis=0)
        
        val_loss = loss_fn.forward(val_logits, y_val)
        val_accuracy = np.mean(val_logits.argmax(1) == y_val)
        
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        print(f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - acc: {train_accuracy:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_accuracy:.4f}")
        
    return history

numpy_model = NumpyCNNMnist(input_shape=(28, 28, 1))
print("Numpy CNN params:", numpy_model.param_count())
# Huấn luyện mô hình NumPy CNN trên 5.000 mẫu để tối ưu thời gian chạy
numpy_history = train_numpy(numpy_model, x_train_np[:5000], y_train_np[:5000], x_val_np[:1000], y_val_np[:1000], epochs=3, batch_size=128)


Numpy CNN params: 52040
Epoch 1/3 - loss: 1.2575 - acc: 0.6604 - val_loss: 0.5813 - val_acc: 0.8210
Epoch 2/3 - loss: 0.3745 - acc: 0.8884 - val_loss: 0.3560 - val_acc: 0.8840
Epoch 3/3 - loss: 0.2552 - acc: 0.9216 - val_loss: 0.2518 - val_acc: 0.9180


In [8]:
# Đánh giá trên tập test NumPy
test_logits = []
for start in range(0, len(y_test_np), 256):
    test_logits.append(numpy_model.forward(x_test_np[start:start+256]))
test_logits = np.concatenate(test_logits, axis=0)

numpy_test_loss = SoftmaxCrossEntropy().forward(test_logits, y_test_np)
numpy_test_accuracy = float(np.mean(test_logits.argmax(1) == y_test_np))
print(f"NumPy Test Accuracy: {numpy_test_accuracy:.4f}")

histories = {'NumPy CNN': numpy_history}
results = [{'model': 'NumPy CNN', 'test_accuracy': numpy_test_accuracy, 'test_loss': numpy_test_loss}]


NumPy Test Accuracy: 0.9281


## 4. TensorFlow/Keras CNN

In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras_train = tf.data.Dataset.from_tensor_slices((x_train_np, y_train_np)).shuffle(10000).batch(128)
keras_val = tf.data.Dataset.from_tensor_slices((x_val_np, y_val_np)).batch(128)
keras_test = tf.data.Dataset.from_tensor_slices((x_test_np, y_test_np)).batch(128)

keras_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

keras_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
keras_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = keras_model.fit(keras_train, validation_data=keras_val, epochs=5, shuffle=False)
histories['Keras CNN'] = history.history

keras_test_loss, keras_test_accuracy = keras_model.evaluate(keras_test, verbose=0)
print(f"Keras Test Accuracy: {keras_test_accuracy:.4f}")

results.append({
    'model': 'Keras CNN', 
    'test_loss': float(keras_test_loss), 
    'test_accuracy': float(keras_test_accuracy),
    'parameters': int(keras_model.count_params())
})


Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 24s 57ms/step - accuracy: 0.9352 - loss: 0.2210 - val_accuracy: 0.9771 - val_loss: 0.0762
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 21s 55ms/step - accuracy: 0.9824 - loss: 0.0583 - val_accuracy: 0.9847 - val_loss: 0.0492
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 57ms/step - accuracy: 0.9882 - loss: 0.0379 - val_accuracy: 0.9849 - val_loss: 0.0487
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 23s 61ms/step - accuracy: 0.9908 - loss: 0.0300 - val_accuracy: 0.9882 - val_loss: 0.0381
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 54ms/step - accuracy: 0.9931 - loss: 0.0219 - val_accuracy: 0.9902 - val_loss: 0.0311
Keras Test Accuracy: 0.9894


## 5. PyTorch CNN

In [11]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Đảo channel: NHWC -> NCHW
Xtr = torch.tensor(x_train_np.transpose(0, 3, 1, 2), dtype=torch.float32)
Ytr = torch.tensor(y_train_np, dtype=torch.long)
Xva = torch.tensor(x_val_np.transpose(0, 3, 1, 2), dtype=torch.float32)
Yva = torch.tensor(y_val_np, dtype=torch.long)
Xte = torch.tensor(x_test_np.transpose(0, 3, 1, 2), dtype=torch.float32)
Yte = torch.tensor(y_test_np, dtype=torch.long)

train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=256)
test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=256)

torch_model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)


Device: cpu


In [12]:
torch_history = {"train_loss":[], "train_accuracy":[], "val_loss":[], "val_accuracy":[]}

for epoch in range(5):
    torch_model.train()
    total_loss = correct = total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = torch_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(yb)
        correct += (out.argmax(1) == yb).sum().item()
        total += len(yb)
        
    train_loss = total_loss / total
    train_accuracy = correct / total
    
    torch_model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = torch_model(xb)
            vl += criterion(out, yb).item() * len(yb)
            vc += (out.argmax(1) == yb).sum().item()
            vt += len(yb)
            
    val_loss = vl / vt
    val_accuracy = vc / vt
    
    torch_history['train_loss'].append(train_loss)
    torch_history['train_accuracy'].append(train_accuracy)
    torch_history['val_loss'].append(val_loss)
    torch_history['val_accuracy'].append(val_accuracy)
    
    print(f"Epoch {epoch+1}/5 - loss: {train_loss:.4f} - acc: {train_accuracy:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_accuracy:.4f}")

histories['PyTorch CNN'] = torch_history


Epoch 1/5 - loss: 0.4057 - acc: 0.8853 - val_loss: 0.1184 - val_acc: 0.9651
Epoch 2/5 - loss: 0.0934 - acc: 0.9722 - val_loss: 0.0702 - val_acc: 0.9783
Epoch 3/5 - loss: 0.0585 - acc: 0.9825 - val_loss: 0.0590 - val_acc: 0.9814
Epoch 4/5 - loss: 0.0461 - acc: 0.9860 - val_loss: 0.0573 - val_acc: 0.9824
Epoch 5/5 - loss: 0.0370 - acc: 0.9889 - val_loss: 0.0483 - val_acc: 0.9843


In [13]:
torch_model.eval()
tl = tc = tt = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = torch_model(xb)
        tl += criterion(out, yb).item() * len(yb)
        tc += (out.argmax(1) == yb).sum().item()
        tt += len(yb)

torch_test_loss = tl / tt
torch_test_accuracy = tc / tt
print(f"PyTorch Test Accuracy: {torch_test_accuracy:.4f}")

results.append({
    'model': 'PyTorch CNN', 
    'test_loss': float(torch_test_loss), 
    'test_accuracy': float(torch_test_accuracy)
})


PyTorch Test Accuracy: 0.9867


## 6. So sánh Kết Quả MNIST

In [14]:
import pandas as pd
df_results = pd.DataFrame(results)
display(df_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(df_results['model'], df_results['test_accuracy'], color=['#4C78A8', '#F58518', '#54A24B'])
axes[0].set_title('So sánh test accuracy MNIST')
axes[0].set_ylim(0, 1)

axes[1].bar(df_results['model'], df_results['test_loss'], color=['#4C78A8', '#F58518', '#54A24B'])
axes[1].set_title('So sánh test loss MNIST')
plt.show()

# Lưu results
df_results.to_csv("mnist_results.csv", index=False)


,model,test_accuracy,test_loss,parameters
0,NumPy CNN,0.9281,0.238763,NaN
1,Keras CNN,0.9894,0.031116,421642.0
2,PyTorch CNN,0.9867,0.039841,NaN
